In [1]:
import numpy as np
from benchmark import (
    generate_datasets,
    generate_num_datasets,
    generate_datasets_with_missing,
    generate_cat_datasets
)  

In [2]:
# gators encoders
from gators.encoders import (
    OneHotEncoder,
    OrdinalEncoder,
    CountEncoder,
    RareCategoryEncoder,
    TargetEncoder,
    WOEEncoder
)

# gators imputers
from gators.imputers import (
    NumericImputer,
    StringImputer
)
# gators discretizers
from gators.discretizers import (
    QuantileDiscretizer,
    EqualLengthDiscretizer,
    CustomDiscretizer,
    TreeBasedDiscretizer,
    GeometricDiscretizer
)

# gators scalers
from gators.scalers import (
    StandardScaler,
    BoxCox,
    YeoJohnson,
    ArcSinhScaler,
    LogScaler,
    PowerScaler,
    ArcSinSquareRootScaler
)

# Feature-engine encoders
from feature_engine.encoding import (
    OneHotEncoder as FEOneHotEncoder,
    OrdinalEncoder as FEOrdinalEncoder,
    CountFrequencyEncoder,
    RareLabelEncoder,
    MeanEncoder,
    WoEEncoder
)

# Feature-engine discretisers
from feature_engine.discretisation import (
    EqualFrequencyDiscretiser,
    EqualWidthDiscretiser,
    ArbitraryDiscretiser,
    # DecisionTreeDiscretiser,
    GeometricWidthDiscretiser
)
# Feature-engine imputers
from feature_engine.imputation import (
    MeanMedianImputer,
    ArbitraryNumberImputer,
    CategoricalImputer
)

# Feature-engine transformers (for scalers)
from feature_engine.transformation import (
    BoxCoxTransformer,
    YeoJohnsonTransformer,
    LogTransformer,
    PowerTransformer,
    ArcsinTransformer,
    ArcSinhTransformer
)
from feature_engine.scaling import MeanNormalizationScaler
from gators.discretizers.equal_length_discretizer import compute_equal_length_bins


from benchmark_streamlined_approach import benchmark_all_transformers

In [3]:
dataset_sizes = [1_000 , 10_000, 100_000, 1_000_000]

cat_datasets = generate_cat_datasets(dataset_sizes)
cat_subset = cat_datasets[1000]["polars"].columns

num_datasets = generate_num_datasets(dataset_sizes)
num_subset = num_datasets[1000]["polars"].columns

datasets = generate_datasets(dataset_sizes)

datasets_missing = generate_datasets_with_missing(dataset_sizes)

Generating 1,000 row dataset...
Generating 10,000 row dataset...
Generating 100,000 row dataset...
Generating 1,000,000 row dataset...
Generating 1,000 row dataset...
Generating 10,000 row dataset...
Generating 100,000 row dataset...
Generating 1,000,000 row dataset...
Generating 1,000 row dataset...
Generating 10,000 row dataset...
Generating 100,000 row dataset...
Generating 1,000,000 row dataset...
Generating 1,000 row dataset...
Generating 10,000 row dataset...
Generating 100,000 row dataset...
Generating 1,000,000 row dataset...


In [4]:
from collections import defaultdict
import polars as pl

num_0_1_datasets = defaultdict(dict)
epsilon = 1e-10  # Small value to ensure strictly > 0

for k, d in num_datasets.items():
    # Pandas normalization to (0, 1] range
    min_val = d["pandas"].min().min()
    max_val = d["pandas"].max().max()
    # First normalize to [0, 1], then scale to (epsilon, 1]
    normalized_pandas = (d["pandas"] - min_val) / (max_val - min_val)
    num_0_1_datasets[k]["pandas"] = normalized_pandas * (1 - epsilon) + epsilon
    
    # Polars normalization to (0, 1] range
    min_val_pl = d["polars"].select(pl.all().min()).to_numpy().min()
    max_val_pl = d["polars"].select(pl.all().max()).to_numpy().max()
    # First normalize to [0, 1], then scale to (epsilon, 1]
    normalized_polars = (d["polars"] - min_val_pl) / (max_val_pl - min_val_pl)
    num_0_1_datasets[k]["polars"] = normalized_polars * (1 - epsilon) + epsilon

In [5]:
encoders_config = {
    'OneHotEncoder': {
        'gators_class': OneHotEncoder,
        'gators_params': {'drop_columns': True},
        'fe_class': FEOneHotEncoder,
        'fe_params': {'drop_last': False, 'ignore_format': True},
        'supervised': False
    },
    'OrdinalEncoder': {
        'gators_class': OrdinalEncoder,
        'gators_params': {'inplace': True},
        'fe_class': FEOrdinalEncoder,
        'fe_params': {'encoding_method': 'arbitrary', 'ignore_format': True},
        'supervised': False
    },
    'CountEncoder': {
        'gators_class': CountEncoder,
        'gators_params': {'inplace': True},
        'fe_class': CountFrequencyEncoder,
        'fe_params': {'encoding_method': 'count', 'ignore_format': True},
        'supervised': False
    },
    'RareCategoryEncoder': {
        'gators_class': RareCategoryEncoder,
        'gators_params': {'min_count': 0.3, 'inplace': True},
        'fe_class': RareLabelEncoder,
        'fe_params': {'tol': 0.05, 'replace_with': 'Rare', 'n_categories': 4, 'ignore_format': True},
        'supervised': False
    },
    'TargetEncoder': {
        'gators_class': TargetEncoder,
        'gators_params': {'inplace': True},
        'fe_class': MeanEncoder,
        'fe_params': {'ignore_format': True},
        'supervised': True  # Needs y
    },
    'WOEEncoder': {
        'gators_class': WOEEncoder,
        'gators_params': {'inplace': True},
        'fe_class': WoEEncoder,
        'fe_params': {'ignore_format': True},
        'supervised': True,  # Needs y
        'skip_high_cardinality': True  # WOE fails with high cardinality due to division by zero
    }
}


num_imputers_config = {
    'MeanImputer': {
        'gators_class': NumericImputer,
        'gators_params': {'strategy': 'mean', 'inplace': True},
        'fe_class': MeanMedianImputer,
        'fe_params': {'imputation_method': 'mean'},
        'column_type': 'numeric',  # Use numeric columns
        'supervised': False
    },
    'MedianImputer': {
        'gators_class': NumericImputer,
        'gators_params': {'strategy': 'median', 'inplace': True},
        'fe_class': MeanMedianImputer,
        'fe_params': {'imputation_method': 'median'},
        'column_type': 'numeric',  # Use numeric columns
        'supervised': False
    },
    'ConstantNumericImputer': {
        'gators_class': NumericImputer,
        'gators_params': {'strategy': 'constant', 'value': 0.0, 'inplace': True},
        'fe_class': ArbitraryNumberImputer,
        'fe_params': {'arbitrary_number': 0.0},
        'column_type': 'numeric',  # Use numeric columns
        'supervised': False
    },
}

string_imputers_config = {
    'Constant_StringImputer': {
        'gators_class': StringImputer,
        'gators_params': {'strategy': 'constant', 'value': 'MISSING', 'inplace': True},
        'fe_class': CategoricalImputer,
        'fe_params': {'imputation_method': 'missing', 'fill_value': 'MISSING', 'ignore_format': True},
        'column_type': 'categorical',  # Use categorical/string columns
        'supervised': False
    }
}

equal_length_bins = compute_equal_length_bins(num_datasets[1000]["polars"], num_bins=10)

discretizers_config = {
    'EqualFrequencyDiscretizer': {
        'gators_class': QuantileDiscretizer,
        'gators_params': {'num_bins': 10, 'inplace': True, 'drop_columns': True},
        'fe_class': EqualFrequencyDiscretiser,
        'fe_params': {'q': 10, 'return_object': False, 'return_boundaries': False},
        'supervised': False
    },
    'EqualWidthDiscretizer': {
        'gators_class': EqualLengthDiscretizer,
        'gators_params': {'num_bins': 10, 'inplace': True, 'drop_columns': True},
        'fe_class': EqualWidthDiscretiser,
        'fe_params': {'bins': 10, 'return_object': False, 'return_boundaries': False},
        'supervised': False
    },
    'CustomDiscretizer': {
        'gators_class': CustomDiscretizer,
        'gators_params': {
            'bins': equal_length_bins,
            'num_bins': 10,
            'inplace': True,
            'drop_columns': True
        },
        'fe_class': ArbitraryDiscretiser,
        'fe_params': {
            'binning_dict': equal_length_bins,
            'return_object': False,
            'return_boundaries': False
        },
        'supervised': False,
        'needs_custom_bins': True  # Flag that this needs bin configuration per dataset
    },
    # 'DecisionTree': {
    #     'gators_class': TreeBasedDiscretizer,
    #     'gators_params': {
    #         'num_bins': 10,
    #         'task': 'classification',
    #         'min_samples_leaf': 10,
    #         'inplace': True,
    #         'drop_columns': True
    #     },
    #     'fe_class': DecisionTreeDiscretiser,
    #     'fe_params': {
    #         'cv': 2,
    #         'random_state': 42,
    #         'regression': False,
    #     },
    #     'supervised': True  # Needs y
    # },
    # 'Geometric': {
    #     'gators_class': GeometricDiscretizer,
    #     'gators_params': {'num_bins': 3, 'inplace': True, 'drop_columns': True},
    #     'fe_class': GeometricWidthDiscretiser,
    #     'fe_params': {'bins': 3},
    #     'supervised': False
    # }
}

scalers_config = {
    'StandardScaler': {
        'gators_class': StandardScaler,
        'gators_params': {'drop_columns': True},
        'fe_class': MeanNormalizationScaler,  # sklearn, not feature-engine
        'fe_params': {},
        'supervised': False,
        'comparison_lib': 'sklearn'
    },
    'PowerScaler': {
        'gators_class': PowerScaler,
        'gators_params': {'power': 2, 'drop_columns': True},
        'fe_class': PowerTransformer,
        'fe_params': {'exp': 2},
        'supervised': False,
        'comparison_lib': 'feature-engine'
    },
    'ArcSinhScaler': {
        'gators_class': ArcSinhScaler,
        'gators_params': {'drop_columns': True},
        'fe_class': ArcSinhTransformer,
        'fe_params': {},
        'supervised': False,
        'comparison_lib': 'feature-engine'
    }
}
scalers_0_1_config = {
    'LogScaler': {
        'gators_class': LogScaler,
        'gators_params': {'drop_columns': True},
        'fe_class': LogTransformer,
        'fe_params': {},
        'supervised': False,
        'positive_only': True,  # Only works with positive values
        'comparison_lib': 'feature-engine'
    },
    'ArcSinSquareRootScaler': {
        'gators_class': ArcSinSquareRootScaler,
        'gators_params': {'drop_columns': True},
        'fe_class': ArcsinTransformer,
        'fe_params': {},
        'supervised': False,
        'comparison_lib': 'feature-engine'
    }
}

In [6]:
set(datasets_missing[1000]["pandas"].dtypes)

{dtype('float64'), dtype('O')}

In [7]:
encoders_results = benchmark_all_transformers(encoders_config, cat_datasets, dataset_sizes)
discretizers_results = benchmark_all_transformers(discretizers_config, num_datasets, dataset_sizes)
scalers_results = benchmark_all_transformers(scalers_config, datasets, dataset_sizes)
scalers_0_1_results = benchmark_all_transformers(scalers_0_1_config, num_0_1_datasets, dataset_sizes)
num_imputers_results = benchmark_all_transformers(num_imputers_config, datasets_missing, dataset_sizes)
string_imputers_results = benchmark_all_transformers(string_imputers_config, datasets_missing, dataset_sizes)



Benchmarking OneHotEncoder
       1,000 rows: 20.81x speedup
      10,000 rows: 187.11x speedup
⚠️  Comparison warmup timed out
⚠️  Comparison run 1/3 timed out after 300s - using timeout for speedup calculation
     100,000 rows: ≥958.34x speedup
⚠️  Comparison warmup timed out
⚠️  Comparison run 1/3 timed out after 300s - using timeout for speedup calculation
    1,000,000 rows: ≥161.45x speedup

Benchmarking OrdinalEncoder
       1,000 rows: 1.05x speedup
      10,000 rows: 2.61x speedup
     100,000 rows: 7.63x speedup
    1,000,000 rows: 7.84x speedup

Benchmarking CountEncoder
       1,000 rows: 0.64x speedup
      10,000 rows: 2.14x speedup
     100,000 rows: 6.51x speedup
    1,000,000 rows: 7.27x speedup

Benchmarking RareCategoryEncoder
       1,000 rows: 7.28x speedup
      10,000 rows: 12.56x speedup
     100,000 rows: 18.92x speedup
    1,000,000 rows: 17.89x speedup

Benchmarking TargetEncoder
       1,000 rows: 3.51x speedup
      10,000 rows: 6.49x speedup
     100,000

/Users/cpoli/Library/Python/3.14/lib/python/site-packages/feature_engine/discretisation/arbitrary.py:191: UserWarning: During the discretisation, NaN values were introduced in the feature(s) linear_0, quadratic_1, exp_2, periodic_3, linear_4, quadratic_5, exp_6, periodic_7, linear_8, quadratic_9, exp_10, periodic_11, linear_12, quadratic_13, exp_14, periodic_15, linear_16, quadratic_17, exp_18, periodic_19, linear_20, quadratic_21, exp_22, periodic_23, linear_24, quadratic_25, exp_26, periodic_27, linear_28, quadratic_29, exp_30, periodic_31, linear_32, quadratic_33, exp_34, periodic_35, linear_36, quadratic_37, exp_38, periodic_39, linear_40, quadratic_41, exp_42, periodic_43, linear_44, quadratic_45, exp_46, periodic_47, linear_48, quadratic_49.
  warnings.warn(
/Users/cpoli/Library/Python/3.14/lib/python/site-packages/feature_engine/discretisation/arbitrary.py:191: UserWarning: During the discretisation, NaN values were introduced in the feature(s) linear_0, quadratic_1, exp_2, peri

     100,000 rows: 8.31x speedup


/Users/cpoli/Library/Python/3.14/lib/python/site-packages/feature_engine/discretisation/arbitrary.py:191: UserWarning: During the discretisation, NaN values were introduced in the feature(s) linear_0, quadratic_1, exp_2, periodic_3, linear_4, quadratic_5, exp_6, periodic_7, linear_8, quadratic_9, exp_10, periodic_11, linear_12, quadratic_13, exp_14, periodic_15, linear_16, quadratic_17, exp_18, periodic_19, linear_20, quadratic_21, exp_22, periodic_23, linear_24, quadratic_25, exp_26, periodic_27, linear_28, quadratic_29, exp_30, periodic_31, linear_32, quadratic_33, exp_34, periodic_35, linear_36, quadratic_37, exp_38, periodic_39, linear_40, quadratic_41, exp_42, periodic_43, linear_44, quadratic_45, exp_46, periodic_47, linear_48, quadratic_49.
  warnings.warn(
/Users/cpoli/Library/Python/3.14/lib/python/site-packages/feature_engine/discretisation/arbitrary.py:191: UserWarning: During the discretisation, NaN values were introduced in the feature(s) linear_0, quadratic_1, exp_2, peri

    1,000,000 rows: 8.37x speedup

Benchmarking StandardScaler
       1,000 rows: 2.20x speedup
      10,000 rows: 3.67x speedup
     100,000 rows: 10.00x speedup
    1,000,000 rows: 15.50x speedup

Benchmarking PowerScaler
       1,000 rows: 4.95x speedup
      10,000 rows: 7.72x speedup
     100,000 rows: 20.49x speedup
    1,000,000 rows: 64.66x speedup

Benchmarking ArcSinhScaler
       1,000 rows: 4.44x speedup
      10,000 rows: 10.09x speedup
     100,000 rows: 13.70x speedup
    1,000,000 rows: 18.26x speedup

Benchmarking LogScaler
       1,000 rows: 4.17x speedup
      10,000 rows: 7.58x speedup
     100,000 rows: 18.27x speedup
    1,000,000 rows: 22.06x speedup

Benchmarking ArcSinSquareRootScaler
       1,000 rows: 3.17x speedup
      10,000 rows: 5.26x speedup
     100,000 rows: 15.09x speedup
    1,000,000 rows: 19.58x speedup

Benchmarking MeanImputer
       1,000 rows: 8.56x speedup
      10,000 rows: 12.37x speedup
     100,000 rows: 60.76x speedup
    1,000,000 rows:

In [11]:
import pandas as pd
df = pd.concat([
    encoders_results,
    discretizers_results,
    scalers_0_1_results,
    scalers_results,
    num_imputers_results,
    string_imputers_results
])

In [ ]:
df.to_csv("benchmark_results.csv", index=False)